# Part 2. Web scraping GDP from Wikipedia
Author: **Kalybay Nurdaulet**

Page: https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)

The table has GDP of every country in millions of USD from IMF, World Bank and UN.
We use `requests` + `BeautifulSoup`.

Result: `data/gdp_raw.csv`

## About the data

**What is nominal GDP:** gross domestic product = the value of all goods and services produced in a country in one year.
"Nominal" means it is in current prices and converted to US dollars with the current exchange rate
(not adjusted for inflation or price levels). We use it as a measure of **how big the economy is**.

**What is in the table:** one row per country or territory, and GDP from three organisations:

| Column on the page | Source | What it is |
|---|---|---|
| Country/Territory | - | name of the country (sometimes with footnotes) |
| IMF (year) | International Monetary Fund, World Economic Outlook | IMF estimate, the newest one |
| World Bank (year) | World Bank | value from the World Bank |
| United Nations (year) | UN Statistics Division | value from the UN |

- All numbers are in **millions of US dollars**.
- The year of each source is written in the column name (IMF is usually an estimate for the current year).
- If a source has only an older value, the year is written in brackets, for example "407,786 (2025)".
- If a source has no data, it is written "— N/a".
- The table also has territories (Hong Kong, Puerto Rico, Greenland...) and a "World" row at the top.

**Why Wikipedia:** the page collects the three official sources in one table, so it is easy to compare them.
The page has a second table with regions (G7, EU...), we do not need it.

In [1]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

os.makedirs("data", exist_ok=True)

URL = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
HEADERS = {"User-Agent": "SIS1 student project"}

### Checking robots.txt
Wikipedia blocks the default python user agent, so we download robots.txt with requests and then give it to the parser.

In [2]:
robots_text = requests.get("https://en.wikipedia.org/robots.txt", headers=HEADERS).text
rp = RobotFileParser()
rp.parse(robots_text.splitlines())
print("can we scrape this page?", rp.can_fetch(HEADERS["User-Agent"], URL))

can we scrape this page? True


### Downloading the page

In [3]:
response = requests.get(URL, headers=HEADERS)
print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
tables = soup.find_all("table", class_="wikitable")
print("number of tables:", len(tables))

200
number of tables: 2


The first table is the one with countries. We take the header row and then go through all other rows.

In [4]:
table = tables[0]
all_rows = table.find_all("tr")

header = []
for cell in all_rows[0].find_all(["th", "td"]):
    header.append(cell.get_text(" ", strip=True))
print(header)

data = []
for row in all_rows[1:]:
    cells = row.find_all(["th", "td"])
    values = [c.get_text(" ", strip=True) for c in cells]
    if len(values) == len(header):
        data.append(values)

gdp = pd.DataFrame(data, columns=header)
print(gdp.shape)
gdp.head(10)

['Country/Territory', 'IMF ( 2026 ) [ 1 ]', 'World Bank (2025) [ 6 ]', 'United Nations (2024) [ 7 ]']
(222, 4)


,Country/Territory,IMF ( 2026 ) [ 1 ],World Bank (2025) [ 6 ],United Nations (2024) [ 7 ]
0,World,"126,295,331","118,350,166","111,565,144"
1,United States,"32,383,920","30,769,700","29,298,000"
2,China [ n 1 ],"20,851,593","19,498,039","18,743,802"
3,Germany,"5,452,858","5,050,923","4,659,929"
4,Japan,"4,379,253","4,435,163","4,026,211"
5,United Kingdom,"4,264,794","4,002,588","3,685,881"
6,India,"4,153,191","3,956,067","3,952,244"
7,France,"3,596,094","3,366,316","3,160,443"
8,Italy,"2,738,164","2,551,557","2,380,825"
9,Russia,"2,656,452","2,561,310","2,173,386"


### Problems in the data

In [5]:
# rows with footnotes [..], N/a or a year in brackets
problems = gdp[gdp.apply(lambda row: row.str.contains(r"N/a|\(|\[").any(), axis=1)]
print(len(problems))
problems.head(10)

47


,Country/Territory,IMF ( 2026 ) [ 1 ],World Bank (2025) [ 6 ],United Nations (2024) [ 7 ]
2,China [ n 1 ],"20,851,593","19,498,039","18,743,802"
22,Taiwan [ n 3 ],"976,719",— N/a,— N/a
30,United Arab Emirates,"621,546","552,325 (2024)","552,325"
41,Hong Kong [ n 4 ],"450,138","427,310","406,863"
45,Pakistan,"407,786 (2025)","407,307","379,120"
57,Cuba,— N/a,"107,352 (2020)","252,063"
58,Ukraine [ n 5 ],"225,337","214,233","190,741"
81,Sri Lanka,"98,964 (2024)","108,825","98,714"
86,Tanzania [ n 6 ],"94,889","90,143","79,236"
98,Macau [ n 7 ],"54,228","52,061","49,467"


## Understanding the data

In [6]:
print("columns:", list(gdp.columns))
print("rows:", len(gdp))
print("first row:", gdp.iloc[0, 0], "(this is the total for the world)")

columns: ['Country/Territory', 'IMF ( 2026 ) [ 1 ]', 'World Bank (2025) [ 6 ]', 'United Nations (2024) [ 7 ]']
rows: 222
first row: World (this is the total for the world)


In [7]:
# how many values every source has
for col in gdp.columns[1:]:
    missing = gdp[col].str.contains("N/a").sum()
    old_year = gdp[col].str.contains(r"\(").sum()
    print(col)
    print("   has value:", len(gdp) - missing, "| N/a:", missing, "| older year in brackets:", old_year)

IMF ( 2026 ) [ 1 ]
   has value: 198 | N/a: 24 | older year in brackets: 7
World Bank (2025) [ 6 ]
   has value: 215 | N/a: 7 | older year in brackets: 28
United Nations (2024) [ 7 ]
   has value: 213 | N/a: 9 | older year in brackets: 0


IMF has the newest numbers but not for every territory, World Bank has the most values.
That is why later we take IMF first and use the others only when IMF is missing.

The biggest and the smallest economies in the table:

In [8]:
gdp.iloc[1:11]

,Country/Territory,IMF ( 2026 ) [ 1 ],World Bank (2025) [ 6 ],United Nations (2024) [ 7 ]
1,United States,"32,383,920","30,769,700","29,298,000"
2,China [ n 1 ],"20,851,593","19,498,039","18,743,802"
3,Germany,"5,452,858","5,050,923","4,659,929"
4,Japan,"4,379,253","4,435,163","4,026,211"
5,United Kingdom,"4,264,794","4,002,588","3,685,881"
6,India,"4,153,191","3,956,067","3,952,244"
7,France,"3,596,094","3,366,316","3,160,443"
8,Italy,"2,738,164","2,551,557","2,380,825"
9,Russia,"2,656,452","2,561,310","2,173,386"
10,Brazil,"2,635,912","2,279,920","2,185,822"


In [9]:
gdp.tail(5)

,Country/Territory,IMF ( 2026 ) [ 1 ],World Bank (2025) [ 6 ],United Nations (2024) [ 7 ]
217,Palau,377,345,310
218,Marshall Islands,342,308,281
219,Nauru,196,176,187
220,Montserrat,— N/a,— N/a,81
221,Tuvalu,65,57,56


The difference between the biggest (United States, ~32 trillion USD) and the smallest (tens of millions USD) is huge,
so in the analysis we will use log10(GDP).

Comparing the three sources for some countries:

In [10]:
gdp[gdp.iloc[:, 0].isin(["United States", "China", "Germany", "Kazakhstan", "Turkey", "Brazil"])]

,Country/Territory,IMF ( 2026 ) [ 1 ],World Bank (2025) [ 6 ],United Nations (2024) [ 7 ]
1,United States,"32,383,920","30,769,700","29,298,000"
3,Germany,"5,452,858","5,050,923","4,659,929"
10,Brazil,"2,635,912","2,279,920","2,185,822"
16,Turkey,"1,640,223","1,597,293","1,323,255"
49,Kazakhstan,"360,456","306,239","291,480"


The numbers are different because the sources are for different years and use a bit different methods,
but the order of countries is almost the same.
China is not in the result: on the page its name is "China [ n 1 ]" (with a footnote), so the exact name does not match.
This is one more reason why names must be cleaned in Part 3.

What has to be cleaned later (Part 3):
- numbers are text with commas: "32,383,920"
- footnotes in names: "China [ n 1 ]"
- missing values are written as "— N/a"
- some numbers have an older year: "407,786 (2025)"
- the first row is "World", it is not a country
- names are not the same as in World Bank ("South Korea" vs "Korea, Rep.")

### File we save

`data/gdp_raw.csv` - the table exactly as it is on the page (all columns are text, not cleaned):

| Column | Meaning |
|---|---|
| Country/Territory | country name |
| IMF (...) | GDP from IMF, millions USD |
| World Bank (...) | GDP from World Bank, millions USD |
| United Nations (...) | GDP from UN, millions USD |

In [11]:
gdp.to_csv("data/gdp_raw.csv", index=False)
print("saved")

saved
